# Figures 8b & 11: Network-wide NTM Simulation Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.metrics import r2_score

plt.rcParams['font.family'] = 'Times New Roman'

# Load Simulation Data

In [ ]:
# Load network-wide simulation results (.mat files from NTM MATLAB code)
# Each contains regional tau concentration over time for 30 hippocampal regions

data_numeric = loadmat('./data/numeric_sim.mat')
data_mlp = loadmat('./data/mlp_sim.mat')
data_sr = loadmat('./data/sr_sim.mat')
data_lr = loadmat('./data/lr_sim.mat')

# Extract regional tau concentration matrices: shape (n_regions, n_timepoints)
N_numeric = data_numeric['model_output'][0,0][0][0,0][0]
N_mlp = data_mlp['model_output'][0,0][0][0,0][0]
N_sr = data_sr['model_output'][0,0][0][0,0][0]
N_lr = data_lr['model_output'][0,0][0][0,0][0]

In [ ]:
# Time axis
x1 = np.arange(0, 0.10025, 0.0025)
x2 = np.arange(0.1050, 0.3005, 0.005)
x3 = np.arange(0.31, 1.01, 0.01)
x = np.concatenate((x1, x2, x3)) * 180  # convert to days

max_index = 54  # ~60 days
max_days = 60
day_inc = 15

# Figure 8b: Regional Tau Concentration Over Time

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 5), sharey=True)

titles = ['LR sNTM', 'SR sNTM', 'MLP sNTM', 'NTM (Ground Truth)']
data_list = [N_lr, N_sr, N_mlp, N_numeric]

for ax_i, (ax, N_data, title) in enumerate(zip(axs, data_list, titles)):
    for i in range(N_data.shape[0]):
        ax.plot(x[:max_index], N_data[i, :max_index])
    ax.set_xlabel('Days', fontsize=13)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(np.arange(0, max_days+1, day_inc))
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

axs[0].set_ylabel(r'[Tau] ($\mu M$)', fontsize=13)

fig.suptitle('Hippocampal subnetwork NTM simulations', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('Figure8b.png', dpi=300, bbox_inches='tight')
plt.show()

# Figure 11: Surrogate Model Accuracy (R² Scatter Plots)

In [ ]:
# Monthly time indices (approximate)
month_indices = [27, 35, 43, 49, 54]  # ~30, 60, 90, 120, 150 days
month_labels = ['Month 1', 'Month 2', 'Month 3', 'Month 4', 'Month 5']

models = {'MLP sNTM': N_mlp, 'SR sNTM': N_sr, 'LR sNTM': N_lr}
model_names = list(models.keys())

fig, axs = plt.subplots(3, 5, figsize=(20, 12))

for row_i, model_name in enumerate(model_names):
    N_model = models[model_name]
    for col_i, (month_idx, month_label) in enumerate(zip(month_indices, month_labels)):
        ax = axs[row_i, col_i]
        
        y_true = N_numeric[:, month_idx]
        y_pred = N_model[:, month_idx]
        
        r2 = r2_score(y_true, y_pred)
        
        ax.scatter(y_true, y_pred, s=20, alpha=0.7)
        
        # Identity line
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.plot(lims, lims, 'r-', linewidth=1)
        
        ax.text(0.05, 0.9, f'$R^2 = {r2:.3f}$', transform=ax.transAxes, fontsize=11)
        
        if row_i == 0:
            ax.set_title(month_label, fontsize=14, fontweight='bold')
        if col_i == 0:
            ax.set_ylabel(model_name, fontsize=13, fontweight='bold')

fig.supxlabel('[Tau] (nM) - Numeric (Ground Truth)\nNTM', fontsize=14)
fig.supylabel('[Tau] (nM) - Machine Learning Surrogate', fontsize=14)
fig.suptitle('Network wide NTM simulation regional tau accuracy', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('Figure11.png', dpi=300, bbox_inches='tight')
plt.show()